In [3]:
MODEL_NAME="gemini-3.6-flash"

In [4]:
import os 
from getpass import getpass
from google import genai
from google.genai import types
print("Import Successful")

Import Successful


In [6]:
API_KEY=os.getenv("GEMINI_API_KEY")
if not API_KEY:
    API_KEY=getpass("Enter Gemini API Key: ")

client=genai.Client(api_key=API_KEY)

print("Gemini API client created successfully")

Enter Gemini API Key:  ········


Gemini API client created successfully


In [7]:
#Verify hosted Gemma model
available_gemini_models=[]

for model in client.models.list():
    model_name=model.name.removeprefix("models/")

    if "gemini" in model_name.lower():
        available_gemini_models.append(model_name)
    
print("Gemini models available to this API key: ")

for name in available_gemini_models:
    print(" -", name)

if not available_gemini_models:
    raise RuntimeError(
        "No hosted Gemma models were returned for this API key/project. "
        "Check the key, project, billing/quota configuration, and regional availability.")

Gemini models available to this API key: 
 - gemini-2.5-flash
 - gemini-2.5-pro
 - gemini-2.5-flash-preview-tts
 - gemini-2.5-pro-preview-tts
 - gemini-flash-latest
 - gemini-flash-lite-latest
 - gemini-pro-latest
 - gemini-2.5-flash-lite
 - gemini-2.5-flash-image
 - gemini-3-flash-preview
 - gemini-3.1-pro-preview
 - gemini-3.1-pro-preview-customtools
 - gemini-3.1-flash-lite-preview
 - gemini-3.1-flash-lite
 - gemini-3-pro-image-preview
 - gemini-3-pro-image
 - gemini-3.1-flash-image-preview
 - gemini-3.1-flash-image
 - gemini-3.1-flash-lite-image
 - gemini-3.5-flash
 - gemini-3.5-flash-lite
 - gemini-omni-flash-preview
 - gemini-omni-1.1-flash
 - gemini-3.5-transcribe
 - gemini-3.6-flash
 - gemini-3.7-flash
 - gemini-3.1-flash-tts-preview
 - gemini-robotics-er-2-preview
 - gemini-2.5-computer-use-preview-10-2025
 - gemini-embedding-001
 - gemini-embedding-2-preview
 - gemini-embedding-2
 - gemini-3.5-transcribe-live
 - gemini-2.5-flash-native-audio-latest
 - gemini-2.5-flash-native-

In [8]:
if MODEL_NAME not in available_gemini_models:
    raise RuntimeError(
        f"{MODEL_NAME!r} is not available to this API key."
        f"Available Gemma models: {available_gemini_models}")

In [12]:
SYSTEM_INSTRUCTION = """
You are a boundary-value test-case generator.

Follow the supplied algorithm exactly. Perform the calculations internally.
Your final response must contain the only one valid JSON object.
Do not output reasoning, Markdown, headings, comments, or code fences."""

USER_PROMPT="""
You are a boundary-value test-case generator.

RULE:
If FICO > 750 and FICO <=900 and
NOINQ>=2 and NOINQ <=99 then
DECISIONCD is DECLINED and 
DECISIONDESC is INQUIRIES

Instructions:
1. Treat variable as Integers.
2. Determine inclusive min.max values.
3. Compute middle=floor(min+max)/2).
4. Generate all combinations. 
5. Return exactly 9 rows.
6. Return JSON only.

"""


In [10]:
response=client.models.generate_content(
    model=MODEL_NAME,
    contents=USER_PROMPT)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[
  {
    "FICO": 751,
    "NOINQ": 2,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 751,
    "NOINQ": 50,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 751,
    "NOINQ": 99,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 825,
    "NOINQ": 2,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 825,
    "NOINQ": 50,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 825,
    "NOINQ": 99,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 900,
    "NOINQ": 2,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 900,
    "NOINQ": 50,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  },
  {
    "FICO": 900,
    "NOINQ": 99,
    "DECISIONCD": "DECLINED",
    "DECISIONDESC": "INQUIRIES"
  }
]


In [14]:
chat = client.chats.create(
    model=MODEL_NAME,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0,
        max_output_tokens=2048,
    ),
)

response = chat.send_message(USER_PROMPT)
print(response.text)

{"test_cases":[{"FICO":751,"NOINQ":2,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":751,"NOINQ":50,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":751,"NOINQ":99,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":825,"NOINQ":2,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":825,"NOINQ":50,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":825,"NOINQ":99,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":900,"NOINQ":2,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":900,"NOINQ":50,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"},{"FICO":900,"NOINQ":99,"DECISIONCD":"DECLINED","DECISIONDESC":"INQUIRIES"}]}


In [19]:
data = json.loads(response.text)

# Print each test case on an individual new line
for case in data["test_cases"]:
  print(case)

{'FICO': 751, 'NOINQ': 2, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 751, 'NOINQ': 50, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 751, 'NOINQ': 99, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 825, 'NOINQ': 2, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 825, 'NOINQ': 50, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 825, 'NOINQ': 99, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 900, 'NOINQ': 2, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 900, 'NOINQ': 50, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
{'FICO': 900, 'NOINQ': 99, 'DECISIONCD': 'DECLINED', 'DECISIONDESC': 'INQUIRIES'}
